# Amazon Beauty Recommender — Two-Tower Training
**在 Colab T4 GPU 上训练两塔召回模型**

运行顺序：从上到下依次运行每个 cell

In [ ]:
# ── Cell 1: 检查 GPU ──────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── Cell 2: 安装依赖 ──────────────────────────────────────────────
!pip install faiss-gpu pandas pyarrow tqdm -q

In [ ]:
# ── Cell 3: Clone 代码 ────────────────────────────────────────────
!git clone https://github.com/96528025/amazon-beauty-rec.git
%cd amazon-beauty-rec

In [ ]:
# ── Cell 4: 下载数据（约 2.8GB，需要 5-10 分钟）────────────────────
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

!curl -L --progress-bar \
  -o data/raw/Beauty_and_Personal_Care.jsonl.gz \
  'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Beauty_and_Personal_Care.jsonl.gz'

In [ ]:
# ── Cell 5: 下载元数据 ────────────────────────────────────────────
!curl -L --progress-bar \
  -o data/raw/meta_All_Beauty.jsonl.gz \
  'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz'

In [ ]:
# ── Cell 6: 数据预处理 ────────────────────────────────────────────
!python src/preprocess.py

In [ ]:
# ── Cell 7: 训练两塔模型 + 建 FAISS 索引 ─────────────────────────
import sys
sys.path.append('.')

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import numpy as np
import faiss
import pandas as pd
from torch.utils.data import DataLoader
from src.two_tower import TwoTowerModel
from src.dataset   import TwoTowerDataset
from src.metrics   import evaluate

DATA_DIR   = 'data/processed'
EMBEDDING_DIM = 64
BATCH_SIZE    = 4096
EPOCHS        = 10
LR            = 1e-3
K             = 10
EVAL_USERS    = 10000

# 加载数据
train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
test  = pd.read_parquet(f'{DATA_DIR}/test.parquet')
num_users = int(train['user_idx'].max()) + 1
num_items = int(train['item_idx'].max()) + 1
print(f'Users: {num_users:,}  Items: {num_items:,}')

# Dataset
dataset = TwoTowerDataset(train, num_items)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
print(f'{len(dataset):,} samples, {len(loader):,} batches/epoch')

# 模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
model     = TwoTowerModel(num_users, num_items, EMBEDDING_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# 训练
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for users, pos_items, neg_items in loader:
        users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)
        optimizer.zero_grad()
        loss = model(users, pos_items, neg_items)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS}  loss={total_loss/len(loader):.4f}')

torch.save(model.state_dict(), f'{DATA_DIR}/two_tower.pt')
print('Model saved!')

In [ ]:
# ── Cell 8: 建 FAISS 索引 + 评估 ─────────────────────────────────
model.eval()
all_item_ids = torch.arange(num_items, device=device)

item_vecs = []
with torch.no_grad():
    for i in range(0, num_items, 4096):
        batch = all_item_ids[i:i+4096]
        vecs  = model.get_item_embedding(batch).cpu().numpy()
        item_vecs.append(vecs)

item_vecs = np.vstack(item_vecs).astype(np.float32)

# FAISS index
res   = faiss.StandardGpuResources()
index_cpu = faiss.IndexFlatIP(EMBEDDING_DIM)
index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
index.add(item_vecs)
print(f'FAISS index: {index.ntotal:,} items')

# 评估
ground_truth  = test.groupby('user_idx')['item_idx'].apply(set).to_dict()
test_user_ids = list(ground_truth.keys())
np.random.seed(42)
sampled_users = np.random.choice(test_user_ids, size=min(EVAL_USERS, len(test_user_ids)), replace=False)
train_user_items = train.groupby('user_idx')['item_idx'].apply(set).to_dict()

with torch.no_grad():
    user_tensor = torch.tensor(sampled_users, dtype=torch.long, device=device)
    user_vecs   = model.get_user_embedding(user_tensor).cpu().numpy().astype(np.float32)

_, top_indices = index.search(user_vecs, K + 50)

recommended = {}
for i, user_idx in enumerate(sampled_users):
    seen = train_user_items.get(int(user_idx), set())
    recs = [item for item in top_indices[i] if item not in seen][:K]
    recommended[int(user_idx)] = recs

sampled_gt = {int(u): ground_truth[u] for u in sampled_users if u in ground_truth}
scores     = evaluate(recommended, sampled_gt, k=K)

print('\n' + '='*50)
print(f'Two-Tower Results (K={K})')
print('='*50)
for metric, value in scores.items():
    print(f'  {metric}: {value}')
print('='*50)
print(f'ALS baseline → Precision@10: 0.0022 | Recall@10: 0.0223')

In [ ]:
# ── Cell 9: 下载模型文件 ──────────────────────────────────────────
# 把训练好的模型下载到本地
from google.colab import files
files.download('data/processed/two_tower.pt')